In [3]:

import torch
import triton
import triton.language as tl



In [4]:

@triton.jit
def fused_ragged_softmax_kernel(
    X,
    LENGTHS,
    Y,
    LSE,
    N: tl.constexpr,
    SX0: tl.constexpr,
    SX1: tl.constexpr,  
    SY0: tl.constexpr,
    SY1: tl.constexpr,
    BLOCK_N: tl.constexpr,
):
    # TASK 1: Identify which row this program owns
    row = tl.program_id(0)
    
        # TASK 2: Create the column index vector
    cols = tl.arange(0,BLOCK_N)

    # TASK 3: Load this row's valid length
    length = tl.load(LENGTHS+row)

    # TASK 4: Calculate the input pointer vector
    x_ptrs = (X+row*SX0+cols*SX1)

    # TASK 5: Construct the load mask
    # Must handle BOTH matrix and row boundaries
    load_mask = ((cols < length) & (cols < N))

    # TASK 6: Load the values safely
    # Choose the correct identity for maximum
    values = tl.load(x_ptrs,mask=load_mask,other= float("-inf"))

    # TASK 7: Find the maximum valid value
    row_max = tl.max(values,axis=0)

    # TASK 8: Subtract maximum and exponentiate
    exp_values = tl.exp(values-row_max) 

    # TASK 9: Calculate the denominator
    denominator = tl.sum(exp_values)

    # TASK 10: Normalize the exponentials
    probabilities = exp_values/denominator

    # TASK 11: Calculate log-sum-exp
    lse =  row_max+tl.log(denominator)

    # TASK 12: Calculate output pointers
    y_ptrs = Y+ row*SY0+cols*SY1

    # TASK 13: Store probabilities
    # All j >= length and j < N must be zero
    tl.store(y_ptrs,probabilities,mask=(cols < N))

    # TASK 14: Store log-sum-exp
    tl.store(LSE+row,lse)
    


In [5]:

def fused_ragged_softmax(x, lengths, y):
    """
    x:       (M, N) float32 CUDA tensor
    lengths: (M,) int32 CUDA tensor
    y:       preallocated (M, N) float32 CUDA tensor

    Returns:
        lse: (M,) float32 CUDA tensor
    """

    M, N = x.shape

    assert x.is_cuda and y.is_cuda
    assert lengths.is_cuda
    assert x.shape == y.shape
    assert lengths.shape == (M,)
    assert x.dtype == torch.float32
    assert y.dtype == torch.float32
    assert lengths.dtype == torch.int32
    assert N >= 1

    lse = torch.empty(
        (M,), device=x.device, dtype=torch.float32
    )

    # TASK 15: Choose a valid BLOCK_N
    BLOCK_N = triton.next_power_of_2(N)

    # TASK 16: Set the launch grid
    grid = (M,)

    # TASK 17: Launch your Triton kernel
    fused_ragged_softmax_kernel[grid](
        x, lengths, y, lse,
        N,
        x.stride(0), x.stride(1),
        y.stride(0), y.stride(1),
        BLOCK_N=BLOCK_N,
    )

    return lse


In [6]:

def run_tests():
    torch.manual_seed(42)
    device = "cuda"

    cases = [
        (3, 5),
        (17, 129),
        (11, 513),
        (8, 1023),
        (3, 2051),
    ]

    passed = 0
    total = 0

    for M, N in cases:
        for layout in ["contiguous", "strided", "transposed"]:
            for extreme in [False, True]:

                if layout == "contiguous":
                    x = torch.randn(M, N, device=device)

                elif layout == "strided":
                    base = torch.randn(M, N * 2, device=device)
                    x = base[:, ::2]

                else:
                    base = torch.randn(N, M, device=device)
                    x = base.T

                if extreme:
                    x = (x * 50 + 10000)
                    # Restore non-contiguous input if needed
                    if layout == "strided":
                        x = torch.stack([x, x], dim=-1)[:, :, 0]
                    elif layout == "transposed":
                        x = x.T.contiguous().T

                lengths = torch.randint(
                    1, N + 1,
                    (M,),
                    device=device,
                    dtype=torch.int32,
                )
                lengths[0] = 1
                lengths[-1] = N

                # Non-contiguous output
                y = torch.empty(
                    (N, M), device=device
                ).T

                total += 1

                try:
                    lse = fused_ragged_softmax(
                        x, lengths, y
                    )

                    cols = torch.arange(
                        N, device=device
                    )

                    valid = cols[None, :] < lengths[:, None]

                    masked_x = x.masked_fill(
                        ~valid, -float("inf")
                    )

                    ref_y = torch.softmax(
                        masked_x, dim=1
                    )
                    ref_lse = torch.logsumexp(
                        masked_x, dim=1
                    )

                    torch.testing.assert_close(
                        y, ref_y,
                        rtol=1e-4, atol=1e-5
                    )
                    torch.testing.assert_close(
                        lse, ref_lse,
                        rtol=1e-4, atol=1e-3
                    )

                    torch.testing.assert_close(
                        y.sum(dim=1),
                        torch.ones(M, device=device),
                        rtol=1e-4, atol=1e-4
                    )

                    assert torch.all(y[~valid] == 0)

                    passed += 1
                    print(f"PASS {M}x{N} {layout} extreme={extreme}")

                except Exception as e:
                    print(f"FAIL {M}x{N} {layout} extreme={extreme}")
                    print(e)

    print(f"\nResult: {passed}/{total} tests passed")


run_tests()


PASS 3x5 contiguous extreme=False
PASS 3x5 contiguous extreme=True
PASS 3x5 strided extreme=False
PASS 3x5 strided extreme=True
PASS 3x5 transposed extreme=False
PASS 3x5 transposed extreme=True
PASS 17x129 contiguous extreme=False
PASS 17x129 contiguous extreme=True
PASS 17x129 strided extreme=False
PASS 17x129 strided extreme=True
PASS 17x129 transposed extreme=False
PASS 17x129 transposed extreme=True
PASS 11x513 contiguous extreme=False
PASS 11x513 contiguous extreme=True
PASS 11x513 strided extreme=False
PASS 11x513 strided extreme=True
PASS 11x513 transposed extreme=False
PASS 11x513 transposed extreme=True
PASS 8x1023 contiguous extreme=False
PASS 8x1023 contiguous extreme=True
PASS 8x1023 strided extreme=False
PASS 8x1023 strided extreme=True
PASS 8x1023 transposed extreme=False
PASS 8x1023 transposed extreme=True
PASS 3x2051 contiguous extreme=False
PASS 3x2051 contiguous extreme=True
PASS 3x2051 strided extreme=False
PASS 3x2051 strided extreme=True
PASS 3x2051 transposed ext